In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from torchvision import datasets, transforms
transform = transforms.Compose([
transforms.ToTensor(),
transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
trainset = datasets.CIFAR10(root='./data', train=True,
download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
shuffle=True, num_workers=2)
testset = datasets.CIFAR10(root='./data', train=False,
download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
shuffle=False, num_workers=2)
# Define the classes in CIFAR-10 dataset
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog',
'horse', 'ship', 'truck')
# Show some example images from the dataset
def imshow(img):

  img = img / 2 + 0.5     #unnormalize

  npimg = img.numpy()
  plt.imshow(np.transpose(npimg, (1, 2, 0)))
  plt.show()
# Get some random training images
  dataiter = iter(trainloader)  # CONFIGURED DATALOADER INSTANCE
  images, labels = next(dataiter)  # explicitly pulls out the next iter

# Show images
  imshow(torchvision.utils.make_grid(images))
# Print labels
  print(' '.join('%5s' % classes[labels[j]] for j in range(4)))

Image Classification Model

In [ ]:
# define the architecture of your neural network
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
  def __init__(self):
    super(SimpleCNN,self).__init__()
    self.conv1 = nn.Conv2d(3,6,5)
    self.pool = nn.MaxPool2d(2,2)
    self.conv2d = nn.Conv2d(6,16,5)
    self.fc1 = nn.Linear(16*5*5,120)
    self.fc2 = nn.Linear(120,84)
    self.fc3 = nn.Linear(84,10)
  def forward(self,x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = x.view(-1, 16 * 5 * 5)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x
net = SimpleCNN()
compiled_net = torch.compile(net)
import torchviz
x = torch.randn(1, 3, 32, 32)  # assuming input size 32x32 RGB
y = net(x)
torchviz.make_dot(y.mean(),
params=dict(net.named_parameters())).render("network_pt1x",
format="png")
y = compiled_net(x)
torchviz.make_dot(y.mean(),params=dict(compiled_net.named_parameters())).render("network_pt2_0", format="png")

In [ ]:
transform = transforms.Compose([
transforms.Resize((224, 224)),
transforms.ToTensor(),
transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
trainset = datasets.CIFAR10(root='./data', train=True,
download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
shuffle=True, num_workers=2)
testset = datasets.CIFAR10(root='./data', train=False,
download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
shuffle=False, num_workers=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(compiled_net.parameters(), lr=0.001,
momentum=0.9)


In [ ]:
for epoch in range(1):  # loop over the dataset multiple times
  running_loss = 0.0
for i, data in enumerate(trainloader, 0):
  inputs, labels = data
  optimizer.zero_grad()
  outputs = net(inputs)
  loss = criterion(outputs, labels)
  loss.backward()
  optimizer.step()
  running_loss += loss.item()
  if i % 2000 == 1999:
    print('[%d, %5d] loss: %.3f' %(epoch + 1, i + 1, running_loss / 2000))

correct = 0
total = 0
with torch.no_grad():
for data in testloader:
    images, labels = data
    outputs = net(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
    print('Accuracy of the network on the 10000 test images:
    100 * correct / total))


In [ ]:
from PIL import Image
from torchvision.transforms import ToTensor
import torchvision.transforms as transforms
image = Image.open('path').convert('RGB')

transform = transforms.Compose([
transforms.Resize((32, 32)),
transforms.ToTensor(),
])
image = transform(image).unsqueeze(0)
# Pass the image through the model
output = compiled_net(image)
# Get the predicted class
_, predicted = torch.max(output.data, 1)
print('Predicted class: %s' % classes[predicted[0]])

finetune by adjusting various params such as lr, batch size and np. of epochs,
dont forget to unsqueeze the transformed image ( like when u are testing on single image cause the model expects batch dimension also)

**Evaluating model performance and and trouble shooting**

In [ ]:
# essential to understand models accuracy and generalisation
# accuracy
correct = 0
total = 0
with torch.no_grad():
for data in testloader:
images, labels = data
outputs = net(images)
_, predicted = torch.max(outputs.data, 1)
total += labels.size(0)
correct += (predicted == labels).sum().item()
print('Accuracy of the network on the 10000 test images: %d %%' %
(100 * correct / total))

# confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
y_pred = []
y_true = []
with torch.no_grad():
  for data in testloader:
    images, labels = data
    outputs = net(images)
    _,predicted = torch.max(outputs.data,1)

outputs - will be the grad tracked form, outputs.data will remove it that grad tracked and make it normal tensor which will be number of  samples tested and their probabilites tensor , from which max will find out the max value and the index where the max was present and it will give that as the alloted label as final prediction

In [ ]:
y_pred.extend(predicted.numpy())
y_true.extend(labels.numpy())
  cm = confusion_matrix(y_true, y_pred)
  sns.heatmap(cm, annot=True, fmt='d')
  plt.ylabel('Actual')
  plt.xlabel('Predicted')
  plt.show()

In [ ]:
# Prepare for metric calculation
all_labels = np.array([])
all_preds = np.array([])
all_outputs = np.array([])
# Test the network on the test data
with torch.no_grad():
  for data in testloader:
    images, labels = data
    outputs = compiled_net(images)
    _, predicted = torch.max(outputs.data, 1)
    # Store predictions, labels and output scores for metrics
    all_labels = np.append(all_labels, labels.numpy())
    all_preds = np.append(all_preds, predicted.numpy())
    all_outputs = np.append(all_outputs, outputs.numpy())
    # Calculate precision, recall, and F1 score
    precision, recall, f1, _ =
    precision_recall_fscore_support(all_labels, all_preds,
    average='weighted')
    print(f' Precision: {precision:.2f}, Recall: {recall:.2f}, F1 Score:
    {f1:.2f}')

Reciever operating curve, ROC-AUC curve , for classification problems at various threshold settings
**its the graph between tpr and fpr**
the axes have the values of tpr and fpr for the range of thresholds over the axes

In [ ]:
all_labels = np.array([])
all_outputs = np.array([])
with torch.no_grad():
  for data in testloader:
    images,labels = data
    outputs = compiled_net(images)
    _,predicted = torch.max(outputs.data,1)
    all_labels = np.append(all_labels,labels.numpy())
    all_outputs = np.append(all_outputs, outputs.numpy())
#onehot encode labels for ROC-AUC
one_hot_labels = np.eye(10)[all_labels.astype(int)]
roc_auc = roc_auc_score(one_hot_labels,all_outputs.reshape(-1,10),
                        multi_class='ovr')
print(f'ROC-AUC:{roc_auc:.2f}')
# Plot ROC curve for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(10):
fpr[i], tpr[i], _ = roc_curve(one_hot_labels[:, i],
all_outputs.reshape(-1, 10)[:, i])
roc_auc[i] = auc(fpr[i], tpr[i])
plt.figure()
for i in range(10):
plt.plot(fpr[i], tpr[i], label=f'ROC curve of class {classes[i]}
(area = {roc_auc[i]:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-AUC curve per class')
plt.legend(loc="lower right")
plt.show()


each point the the curve represents are threshold where there is that combination of tpr and fpr, so if the mode is having many points where tpr is high and fpr is low means the model is good overall , when compare to a model which has one random threshold where the (tpr,fpr) is good to measure which we find the area under it
**Area is a measure of how much of the threshold space your model performs well on.**